<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Lab 02 — Write-Policy: Extract → Salience → Reconcile

## Learning goals

1. Treat a memory write as **reconciliation**, not a blind insert.
2. Implement the four resolutions: **ADD / UPDATE / DELETE(tombstone) / NOOP**.
3. Separate the **append-log** (lossless ground truth) from the **recall store** (salient facts only).
4. Prefer **background** consolidation over hot-path extraction when latency matters.
5. Assign **importance** $[0, 1]$ at write time so later retrieval (Lab 02b) has something to spend.

## Theory you need

A memory store is only as good as what you chose to put in it. The default failure is not *too little* memory but *too much of the wrong kind* — transient chatter, near-duplicates, and unresolved contradictions ("likes tea" sitting next to "prefers coffee now").

A healthy write-policy has two axes:

| Axis | Question | Rule of thumb |
|------|----------|----------------|
| **When** | Hot-path (during the turn) or background (after)? | Most writing should be background; hot-path is the exception you justify. |
| **What** | Is this worth recalling three sessions from now? | If not, keep it only in the append-log. How much? Score importance $0$–$1$ at extract time ($0.5$ = medium). |

Pipeline (per turn):

```
raw turn  →  extract atomic facts + importance  →  salience gate  →  reconcile vs neighbours
                                                                    ├─ NOOP
                                                                    ├─ UPDATE (tombstone + ADD successor)
                                                                    ├─ DELETE (tombstone + ADD successor)
                                                                    └─ ADD
```

On change or contradiction: **invalidate with a timestamp, do not erase.** Supersession leaves a scar — including UPDATE — so "liked tea" remains visible after a later coffee preference; silent in-place rewrite lies about history.


In [2]:
import json
import re

In [3]:
# Supporting Python lives in src/memory (installed via `uv sync`).
# Notebooks only contain the lab narrative and exercises.

from google.adk.tools import ToolContext

from memory import (
    FactStore,
    complete,
    create_session,
    load_lab_env,
    make_agent,
    make_model,
    make_runner,
    model_summary,
    run_turn,
)

load_lab_env()
print("LLM:", model_summary())
store = FactStore()
append_log: list[dict] = []  # lossless ground truth — never edited in place
print("Fresh FactStore + append-log ready.")


LLM: endpoint=http://10.0.10.51:8000/v1  model=openai/gpt-oss-120b
Fresh FactStore + append-log ready.


## Part A — The extractor (LLM) and the salience gate

We ask the model for **atomic, standalone** facts with pronouns already resolved — and an **importance** score in $[0, 1]$ judged at write time. A memory that says "she said yes" is useless three sessions later; a store where every fact is `0.5` cannot power Lab 02b's retrieval ranking.


In [4]:
EXTRACT_PROMPT = """
You extract durable personal facts from a dialogue turn for a long-term memory store.

Rules:
- Return ONLY a JSON array of objects:
  {{"text": "<atomic fact>", "importance": <float 0.0-1.0>}}
- Each text must be atomic and self-contained (resolve pronouns to names).
- Keep stable preferences, identity facts, constraints, and decisions.
- DROP pleasantries, one-off clarifications, and pure small-talk.
- If nothing durable is present, return [].

Importance scale (float):
- 0.0 = no lasting value (prefer dropping instead of emitting)
- 0.5 = medium — ordinary durable preference or identity detail
- 1.0 = highest — safety/medical constraints, hard requirements, critical identity
Score continuously; allergies/constraints near 0.9–1.0, seat/diet prefs near 0.5–0.7,
soft likes near 0.3–0.5.

Turn:
"{turn}"

User name for pronoun resolution: {user_name}
"""


def parse_json_payload(text: str):
    """Best-effort JSON parse; tolerates fenced markdown from the model."""
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
        if not m:
            return None
        return json.loads(m.group(0))


def salience_ok(fact: str) -> bool:
    """Heuristic stand-in for 'would this matter three sessions from now?'"""
    ephemeral = ("hello", "thanks", "thank you", "how are you", "lol", "ok", "okay")
    low = fact.lower()
    return not (any(tok in low for tok in ephemeral) and len(low.split()) < 6)


def _clamp_importance(value, default: float = 0.5) -> float:
    try:
        score = float(value)
    except (TypeError, ValueError):
        return default
    return max(0.0, min(1.0, score))


def normalize_extracted(item) -> dict | None:
    """Coerce a JSON element to {text, importance}; default importance 0.5."""
    if isinstance(item, dict):
        text = str(item.get("text") or item.get("fact") or "").strip()
        importance = _clamp_importance(item.get("importance"), default=0.5)
    else:
        text = str(item).strip()
        importance = 0.5
    if not text:
        return None
    return {"text": text, "importance": importance}


def extract_facts(turn: str, user_name: str = "Ada") -> list[dict]:
    """Return [{text, importance}, ...] after salience filtering."""
    raw = complete(EXTRACT_PROMPT.format(turn=turn, user_name=user_name), max_tokens=512)
    data = parse_json_payload(raw) or []
    if not isinstance(data, list):
        return []
    facts = []
    for item in data:
        fact = normalize_extracted(item)
        if fact and salience_ok(fact["text"]):
            facts.append(fact)
    return facts


# Smoke test
demo_turn = "Hi! By the way I am vegetarian and I prefer aisle seats on long flights."
facts = extract_facts(demo_turn)
print("Extracted:")
for f in facts:
    print(f"  [{f['importance']:.2f}] {f['text']}")


Extracted:
  [0.80] Ada is vegetarian
  [0.55] Ada prefers aisle seats on long flights


## Part B — Reconcile against neighbours

For each candidate fact we retrieve near neighbours with a **sentence-transformer** (top-k by cosine similarity) and ask the LLM to pick an op. Keyword overlap misses contradictions like "drinks tea" vs "switched to coffee". We also retry empty reconcile replies and run a supersession guard when ADD is proposed against a near neighbour — otherwise a flaky resolve silently ADDs and leaves both facts active.


In [5]:
RESOLVE_PROMPT = """
You reconcile a new candidate fact with existing memories.

Return ONLY a JSON object with keys:
  "op": one of "ADD", "UPDATE", "DELETE", "NOOP"
  "target_id": string or null  (required for UPDATE/DELETE — must be an id from the neighbour list)
  "reason": short string

Meanings:
- NOOP: candidate already entailed by an existing memory.
- UPDATE: candidate refines an existing memory — we tombstone target_id then ADD the refined fact (prior wording kept for audit).
- DELETE: candidate contradicts target_id — same apply path: tombstone target_id then ADD the candidate.
- ADD: genuinely new (orthogonal to all neighbours).

Hard rules:
- Mutually exclusive preferences MUST supersede (UPDATE or DELETE), never ADD alongside.
  Examples: tea vs coffee, city A vs city B, aisle vs window seat.
- If similarity is high and the topic matches, prefer DELETE/UPDATE over ADD.
- Never invent a target_id; copy an id exactly from the neighbour list.

Candidate:
{candidate}

Existing neighbours (id :: similarity :: text):
{neighbours}
"""


def _same_memory_slot(a: str, b: str) -> bool:
    """Heuristic: do two facts claim competing values for the same attribute?"""
    slots = {
        "beverage": {"tea", "coffee", "latte", "espresso", "matcha", "chai"},
        "city": {"live", "lives", "living", "moved", "reside", "resides"},
        "seat": {"aisle", "window", "middle", "seat", "seats"},
        "diet": {"vegetarian", "vegan", "pescatarian", "allergy", "allergic"},
    }
    al, bl = a.lower(), b.lower()
    for words in slots.values():
        if any(w in al for w in words) and any(w in bl for w in words):
            return True
    return False


def parse_resolve(raw: str) -> dict | None:
    data = parse_json_payload(raw)
    if not isinstance(data, dict) or not data.get("op"):
        return None
    data["op"] = str(data.get("op", "ADD")).upper()
    tid = data.get("target_id")
    if tid is not None:
        data["target_id"] = str(tid).strip().split()[0].split(":")[0]
    return data


def resolve(candidate: str, scored_neighbours: list) -> dict:
    if not scored_neighbours:
        return {"op": "ADD", "target_id": None, "reason": "no neighbours"}
    rendered = "\n".join(
        f"{f.id} :: {score:.3f} :: {f.text}" for score, f in scored_neighbours
    )
    prompt = RESOLVE_PROMPT.format(candidate=candidate, neighbours=rendered)
    # gpt-oss spends many tokens on hidden reasoning; keep max_tokens high and retry.
    for i in range(3):
        raw = complete(prompt, max_tokens=1024)
        data = parse_resolve(raw)
        if data is not None:
            return data
        print(f"retrying {i}")
    return {"op": "ADD", "target_id": None, "reason": "parse-fallback"}


def _match_target(target_id: str | None, neighbours: list) -> str | None:
    if not target_id:
        return None
    ids = {n.id for n in neighbours}
    if target_id in ids:
        return target_id
    for n in neighbours:
        if target_id.startswith(n.id) or n.id.startswith(target_id):
            return n.id
    return None


def write_memory(turn: str, *, user_name: str = "Ada") -> list[dict]:
    """Full write-policy: log → extract → salience → reconcile → apply."""
    append_log.append({"turn": turn, "user": user_name})
    candidates = extract_facts(turn, user_name=user_name)
    actions = []
    for candidate in candidates:
        fact = candidate["text"]
        importance = candidate["importance"]
        scored = store.search_scored(fact, k=5)
        neighbours = [f for _, f in scored]
        decision = resolve(fact, scored)
        op = decision["op"]
        target_id = _match_target(decision.get("target_id"), neighbours)

        # Bind garbled UPDATE/DELETE target ids to the nearest neighbour.
        if op in ("UPDATE", "DELETE") and target_id is None and scored:
            target_id = scored[0][1].id
            decision["reason"] = (
                f"{decision.get('reason', '')}; bound to nearest neighbour {target_id}"
            ).strip("; ")

        # Never leave two active facts in the same slot (tea+coffee). Covers empty
        # reconcile JSON (parse-fallback) and resolve wrongly choosing ADD.
        if op == "ADD" and scored and _same_memory_slot(fact, scored[0][1].text):
            print(f"same-slot supersede vs {scored[0][1].id} (sim={scored[0][0]:.3f})")
            best_score, best = scored[0]
            op = "DELETE"
            target_id = best.id
            decision = {
                "op": "DELETE",
                "target_id": target_id,
                "reason": (
                    f"same-slot supersede vs {best.id} (sim={best_score:.3f}); "
                    f"prior={decision.get('reason', '')}"
                ),
            }

        if op == "NOOP":
            actions.append({"fact": fact, "importance": importance, **decision})
        elif op in ("UPDATE", "DELETE") and target_id is not None:
            store.update(
                target_id, fact, importance=importance, provenance=turn
            )
            decision["target_id"] = target_id
            actions.append({"fact": fact, "importance": importance, **decision})
        else:
            store.add(fact, importance=importance, provenance=turn)
            actions.append(
                {
                    "fact": fact,
                    "importance": importance,
                    "op": "ADD",
                    "target_id": None,
                    "reason": decision.get("reason", ""),
                }
            )
    return actions

print("write_memory() ready (sync — safe to call from ADK tools).")


write_memory() ready (sync — safe to call from ADK tools).


## Part C — Drive the policy with an ADK agent (hot-path tool)

The agent talks to the user; a tool triggers consolidation for the latest turn. In production you would usually defer this to a callback / background worker (**write-behind**). We keep it as a tool so the notebook stays interactive.


In [6]:
# Shared handle so the tool closes over the latest user text.
PENDING_TURN = {"text": ""}

def consolidate_latest_turn(tool_context: ToolContext) -> dict:
    """Run the write-policy on the latest user turn and mirror durable facts into user: state."""
    turn = PENDING_TURN["text"]
    if not turn:
        return {"ok": False, "reason": "no pending turn"}
    actions = write_memory(turn)
    active = [f.text for f in store.all()]
    tool_context.state["user:memory_facts"] = active
    return {"ok": True, "actions": actions, "active_facts": active}


agent = make_agent(
    name="memory_writer",
    model=make_model(),
    instruction=(
        "You are a helpful assistant. After acknowledging the user, "
        "ALWAYS call consolidate_latest_turn exactly once so durable facts are stored. "
        "Then briefly list what changed (ADD/UPDATE/DELETE/NOOP). "
        "Do not narrate chain-of-thought; keep the user-visible reply short."
    ),
    tools=[consolidate_latest_turn],
)
runner, sessions = make_runner(agent, app_name="lab02_write_policy")


async def chat(user_id: str, session_id: str, text: str):
    PENDING_TURN["text"] = text
    reply = run_turn(runner, user_id=user_id, session_id=session_id, message=text)
    print("User:", text)
    print("Agent:", reply)
    print("Store:")
    for f in store.all(include_superseded=True):
        flag = " [TOMBSTONE]" if f.superseded else ""
        print(f"  - {f.id}: {f.text}{flag}")
    print("-" * 60)


sess = await create_session(sessions, app_name="lab02_write_policy", user_id="ada")
await chat("ada", sess.id, "Hi there, thanks for helping!")
await chat("ada", sess.id, "I drink tea every morning and I live in Austin.")
await chat("ada", sess.id, "Actually I switched to coffee in June 2026 — please remember that.")
await chat("ada", sess.id, "I live in Austin.")  # should trend toward NOOP
await chat("ada", sess.id, "I'm allergic to peanuts.")  # should ADD a new fact on allergy to peanuts


User: Hi there, thanks for helping!
Agent: We need to acknowledge user, then call consolidate_latest_turn, then list changes. No other content.Hi! Happy to help.  

Changes: NOOP.
Store:
------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

User: I drink tea every morning and I live in Austin.
Agent: We need to call consolidate_latest_turn again to store durable facts. The user gave two facts: "I drink tea every morning" and "I live in Austin". We need to store them as durable facts. The system expects we call consolidate_latest_turn which will process the latest turn and mirror durable facts into user: state. We need to call it with no arguments. Then we need to output short reply and list changes: likely ADD for two facts. So after calling, we need to output "Got it!" and list changes: ADD. Probably "ADD: drink tea every morning, live in Austin". Let's do that.Got it!  

Changes: ADD – “drink tea every morning”, “live in Austin”.
Store:
  - 315f3b01: Ada drinks tea every morning
  - d64098f5: Ada lives in Austin
------------------------------------------------------------
User: Actually I switched to coffee in June 2026 — please remember that.
Agent: We need to update the fact about drinking tea to coffee, with date. So

## Part D — Inspect the append-log vs the recall store

The append-log kept every turn. The recall store should be smaller, de-duplicated, honest about supersession, and carry **non-default importances** from the extractor (allergies high, soft prefs nearer the middle).


In [7]:
print(f"append-log turns: {len(append_log)}")
for row in append_log:
    print(" ", row)

print(f"\nrecall store (active): {len(store.all())}")
print(store.render_for_prompt(store.all()))

print(f"\nrecall store (with tombstones): {len(store.all(include_superseded=True))}")
for f in store.all(include_superseded=True):
    print(f.to_dict())

active_texts = " ".join(f.text.lower() for f in store.all())
assert "coffee" in active_texts, "current preference should mention coffee"
assert not any("tea" in f.text.lower() for f in store.all()), (
    "superseded tea preference should not remain active (UPDATE/DELETE both tombstone)"
)
# Prior tea wording should remain as a tombstone for audit:
tombstoned = [f for f in store.all(include_superseded=True) if f.superseded]
assert any("tea" in f.text.lower() for f in tombstoned), "tea preference should remain as tombstone history"
print(f"\nTombstones present: {len(tombstoned)}")
print("✓ Write-policy produced a conflict-aware store.")


append-log turns: 5
  {'turn': 'Hi there, thanks for helping!', 'user': 'Ada'}
  {'turn': 'I drink tea every morning and I live in Austin.', 'user': 'Ada'}
  {'turn': 'Actually I switched to coffee in June 2026 — please remember that.', 'user': 'Ada'}
  {'turn': 'I live in Austin.', 'user': 'Ada'}
  {'turn': "I'm allergic to peanuts.", 'user': 'Ada'}

recall store (active): 3
- (d64098f5) Ada lives in Austin
- (82fc8636) Ada switched to coffee in June 2026
- (efa9841d) Ada is allergic to peanuts.

recall store (with tombstones): 4
{'text': 'Ada lives in Austin', 'id': 'd64098f5', 'importance': 0.6, 'created_at': '2026-07-27T17:48:56.394567+00:00', 'updated_at': '2026-07-27T17:48:56.394577+00:00', 'provenance': 'I drink tea every morning and I live in Austin.', 'superseded': False, 'superseded_by': None, 'superseded_at': None}
{'text': 'Ada switched to coffee in June 2026', 'id': '82fc8636', 'importance': 0.6, 'created_at': '2026-07-27T17:49:00.233438+00:00', 'updated_at': '2026-07-27T1

## Next lab

**Lab 03 — Compaction** deals with the *working* tier: what happens when the live conversation itself no longer fits the window.
